In [ ]:
import os
import numpy as np

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Initialize LLM Client
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# 2. Knowledge Base
documents = [
    {
        "id": "HR-001",
        "title": "Annual Leave Policy",
        "text": "Employees receive 26 annual leave days.",
    },
    {
        "id": "HR-002",
        "title": "Leave Approval Policy",
        "text": "Leave requests must be approved by the manager.",
    },
    {
        "id": "HR-003",
        "title": "Health Insurance Policy",
        "text": "Employees have supplementary health insurance.",
    },
    {
        "id": "HR-004",
        "title": "Salary Policy",
        "text": "Monthly salaries are paid at the end of each month.",
    },
    {
        "id": "HR-005",
        "title": "Remote Work Policy",
        "text": "Employees can work remotely two days per week.",
    },
]

# 3. Load Embedding Model
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

document_embeddings = embedding_model.encode(
    [doc["text"] for doc in documents], convert_to_numpy=True, normalize_embeddings=True
)


# 4. Retrieve Relevant Chunks
def retrieve_chunks(query, top_k=3, threshold=0.45):

    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, document_embeddings)[0]

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for index in ranked_indices:
        score = float(scores[index])

        if score < threshold:
            continue

        results.append(
            {
                "id": documents[index]["id"],
                "title": documents[index]["title"],
                "text": documents[index]["text"],
                "score": score,
            }
        )

        if len(results) >= top_k:
            break

    return results


# 5. Construct Context
def construct_context(chunks):

    return "\n\n".join(f"[Source: {chunk['id']}]\n{chunk['text']}" for chunk in chunks)


# 6. Generate Answer
def generate_answer(query, context):

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": """
You are an HR assistant.

Answer using only the provided context.
Do not invent information.
If the context does not contain the answer,
clearly say that the information is unavailable.
""",
            },
            {
                "role": "user",
                "content": f"""
Context:
{context}

Question:
{query}
""",
            },
        ],
    )

    return response.choices[0].message.content


# 7. Complete RAG Function
def ask_hr_assistant(query):

    retrieved_chunks = retrieve_chunks(query=query, top_k=3, threshold=0.45)

    # Handle No-Result Queries
    if not retrieved_chunks:
        return {
            "answer": "I couldn't find relevant information in the company documents.",
            "sources": [],
            "status": "no_result",
        }

    context = construct_context(retrieved_chunks)

    answer = generate_answer(query=query, context=context)

    sources = [
        {"id": chunk["id"], "title": chunk["title"], "score": chunk["score"]}
        for chunk in retrieved_chunks
    ]

    return {"answer": answer, "sources": sources, "status": "success"}


# 8. Test Queries
queries = [
    "How many annual leave days do employees receive?",
    "What is the company's stock investment policy?",
]

for query in queries:
    result = ask_hr_assistant(query)

    print("\nQUESTION:")
    print(query)

    print("\nSTATUS:")
    print(result["status"])

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")
    for source in result["sources"]:
        print(source)

    print("=" * 60)